# SDR Linkbudget (Student)

Aufgaben:
1. Traeger um 433 MHz messen.
2. Mit `../03_02_LABOR-2/calibration.json` die Empfangsleistung bestimmen.
3. Bei 0 dBi Sende-/Empfangsantenne C/N0 berechnen.
4. Aus Noise Floor die effektive Rauschtemperatur ableiten.
5. LOS-Freiraumdaempfung aus geschaetztem Abstand und Wellenlaenge berechnen.
6. Mit gemessener Empfangsleistung (bei bekannter Sendeleistung) vergleichen und Multipath diskutieren.

Hinweis: Die TODO-Stellen sind Platzhalter. Das Notebook laeuft trotzdem fehlerfrei.

In [ ]:
import math
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

LAB_DIR = Path.cwd()
CAL_PATH = LAB_DIR.parent / "03_02_LABOR-2" / "calibration.json"

# Bekannt gegebene Sendeleistung (vom Versuchsaufbau)
TX_POWER_DBM = 10.0

# TODO: am Geraet gemessene/gesetzte Traegerfrequenz einsetzen
carrier_freq_hz = 433.92e6

# TODO: geschaetzten Abstand Sender-Empfaenger einsetzen
distance_guess_m = 1000.0

print("Calibration file:", CAL_PATH)

In [ ]:
# Kalibrierung laden (wird fuer echte Messung benoetigt)
cal = json.loads(CAL_PATH.read_text(encoding="utf-8"))
print("Reference calibration frequency:", cal.get("reference_freq_hz", "n/a"), "Hz")

# TODO (Messung):
# - Spektrum um 433 MHz aufnehmen
# - Peak finden
# - Mit calibration_offset_dB auf dBm kalibrieren
# - Noise floor aus benachbarten Bins bestimmen

# Platzhalterwerte (damit Notebook laeuft)
rx_power_dbm = -95.0
noise_bin_dbm = -120.0
bin_bw_hz = 500.0

print(f"RX power placeholder      : {rx_power_dbm:.2f} dBm")
print(f"Noise floor placeholder   : {noise_bin_dbm:.2f} dBm / bin")
print(f"Bin bandwidth placeholder : {bin_bw_hz:.2f} Hz")

In [ ]:
# C/N0 und effektive Rauschtemperatur

# TODO: N0 aus Noise-Bin auf dBm/Hz normieren
n0_dbm_hz = noise_bin_dbm - 10.0 * math.log10(bin_bw_hz)

# TODO: C/N0 = C - N0
cn0_db_hz = rx_power_dbm - n0_dbm_hz

# TODO: T_eff aus N0 ableiten, Referenz -174 dBm/Hz bei 290 K
t_eff_k = 290.0 * 10.0 ** ((n0_dbm_hz + 174.0) / 10.0)

print(f"N0                : {n0_dbm_hz:.2f} dBm/Hz")
print(f"C/N0              : {cn0_db_hz:.2f} dB-Hz")
print(f"T_eff             : {t_eff_k:.1f} K")

In [ ]:
# Freiraumdaempfung (LOS) und Vergleich zur Messung

c0 = 299_792_458.0

# TODO: Wellenlaenge aus gemessener Traegerfrequenz
wavelength_m = c0 / carrier_freq_hz

# Gemessene Linkdaempfung bei 0 dBi / 0 dBi
fspl_meas_db = TX_POWER_DBM - rx_power_dbm

# TODO: theoretische LOS-Freiraumdaempfung mit geschaetztem Abstand
fspl_los_db = 20.0 * math.log10(4.0 * math.pi * distance_guess_m / wavelength_m)

# TODO: Differenz als Multipath-Indiz interpretieren
delta_db = fspl_meas_db - fspl_los_db

print(f"Wavelength        : {wavelength_m:.4f} m")
print(f"FSPL measured     : {fspl_meas_db:.2f} dB")
print(f"FSPL LOS (guess)  : {fspl_los_db:.2f} dB")
print(f"Delta             : {delta_db:+.2f} dB")

In [ ]:
# Einfache Visualisierung der wichtigsten Kennwerte
labels = ["RX dBm", "N0 dBm/Hz", "C/N0 dB-Hz", "FSPL_meas dB", "FSPL_LOS dB"]
values = [rx_power_dbm, n0_dbm_hz, cn0_db_hz, fspl_meas_db, fspl_los_db]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, values, color=["C0", "C1", "C2", "C3", "C4"])
ax.set_title("Linkbudget Uebersicht (mit Platzhalterwerten)")
ax.grid(True, axis="y", alpha=0.3)
for i, v in enumerate(values):
    ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

print("TODO: Platzhalterwerte durch reale Messwerte ersetzen.")